In [1]:
import copy
from pprint import pprint

import numpy as np
import pandas as pd
from parse import parse
from pathlib import Path

In [2]:
from ase.data import chemical_symbols

# Create the dictionary using a dict comprehension
num_to_symbol = {i: symbol for i, symbol in enumerate(chemical_symbols) if i > 0}

# Example lookup
#print(num_to_symbol[6])   # Output: 'C'
#print(num_to_symbol[12])  # Output: 'Mg'

In [3]:
def parse_grrm_saddlepoint_irc_log(log_file_path: Path, num_atoms = 12) -> dict:
    """This function parses a single log file from a specific assumed set of input parameters in a .com file submitted to GRRM.

    See the `assemble_com()` function for the pattern of the .com file which is assumed to be the basis for the log file to be
    parsed by this function. 

    
    """
    log_parsing_results = {"initial_TS_energy": None,
                           "initial_TS_geometry": None,
                           "saddle_point_found": False,
                           "reopt_TS_energy": None,
                           "reopt_TS_geometry": None,
                           "reopt_TS_num_steps": None,
                           "reopt_TS_frequencies": None,
                           "reopt_TS_single_neg_freq_confirmed": False,
                           "1st_irc_EQ_energy": None,
                           "1st_irc_EQ_geometry": None,
                           "2nd_irc_EQ_energy": None,
                           "2nd_irc_EQ_geometry": None,
                          }
    
    itr_header_pattern = "# ITR. {iter_num}"
    consuming_coordinates = False
    start_consuming_coordinates_at = -1 
    stop_consuming_coordinates_at = -1 
    consuming_energy = False
    consume_energy_at = -1
    saddle_point_found = False
    optimized_structure = False
    consuming_freq = False
    reopt_TS_frequencies = []
    
    # With saddle point optimization AND IRC, we have three stages of the log:
    # Stage 1: re-optimization of candidate TS to true saddle point.
    # Stage 2: first EQ-finding by IRC.
    # Stage 3: second EQ-finding by IRC.
    
    stage = 1 
    
    with open(log_file_path, "r") as f: 
        for i, line in enumerate(f):
            if line.startswith("# ITR."):
                iter_parsed = parse(itr_header_pattern, line)
                iter_num = int(iter_parsed["iter_num"])
                if stage == 1 and iter_num == 0:
                    consuming_energy = True
                    consume_energy_at = i + num_atoms + 2
                    consuming_coordinates = True
                    start_consuming_coordinates_at = i+1
                    stop_consuming_coordinates_at = i+num_atoms
                    temp_coords = []
            elif line.startswith("Optimized structure"):
                optimized_structure = True
                consuming_energy = True
                consume_energy_at = i + num_atoms + 1
                consuming_coordinates = True
                start_consuming_coordinates_at = i+1
                stop_consuming_coordinates_at = i+num_atoms
                temp_coords = []
            elif line.startswith("1st-Order Saddle point was found"):
                saddle_point_found = True
                log_parsing_results["saddle_point_found"] = saddle_point_found
                print(f"Saddle point was found after {iter_num} iterations.")
            elif saddle_point_found and line.startswith("FREQFREQFREQ") and len(reopt_TS_frequencies) == 0:
                #print("Toggling frequency parsing on.")
                consuming_freq = True
            elif (consuming_freq) and (line.startswith("Freq.  :")):
                temp_freqs = [float(x) for x in line.rstrip().split()[-3:]]
                for x in temp_freqs:
                    reopt_TS_frequencies.append(x)
                if len(reopt_TS_frequencies) == 30:
                    #print("Toggling frequency parsing off.\n")
                    consuming_freq = False
                    log_parsing_results["reopt_TS_frequencies"] = copy.deepcopy(reopt_TS_frequencies)
                    if reopt_TS_frequencies[0] < 0.0 and all([x >= 0.0 for x in reopt_TS_frequencies[1:]]):
                        log_parsing_results["reopt_TS_single_neg_freq_confirmed"] = True
            if consuming_coordinates and start_consuming_coordinates_at <= i:
                temp_coord_row = [line.rstrip().split()[0]] + [float(x) for x in line.rstrip().split()[1:]]
                temp_coords.append(temp_coord_row)
                if stop_consuming_coordinates_at == i:
                    consuming_coordinates = False
                    if stage == 1 and iter_num == 0:
                        log_parsing_results["initial_TS_geometry"] = copy.deepcopy(temp_coords)
                    elif optimized_structure:
                        if stage == 1:
                            log_parsing_results["reopt_TS_geometry"] = copy.deepcopy(temp_coords)
                        elif stage == 2:
                            log_parsing_results["1st_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
                        elif stage == 3:
                            log_parsing_results["2nd_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
            if consuming_energy and consume_energy_at == i:
                if stage == 1 and iter_num == 0:
                    log_parsing_results["initial_TS_energy"] = float(line.split()[1])
                elif optimized_structure:
                    print("Found optimized structure and energy.")
                    temp_energy = float(line.rstrip().split()[2])
                    if stage == 1:
                        log_parsing_results["reopt_TS_energy"] = temp_energy
                        log_parsing_results["reopt_TS_num_steps"] = iter_num
                    elif stage == 2:
                        log_parsing_results["1st_irc_EQ_energy"] = temp_energy
                    elif stage == 3:
                        log_parsing_results["2nd_irc_EQ_energy"] = temp_energy
                    optimized_structure = False
                    if stage <= 2:
                        stage += 1
                consuming_energy = False
                consume_energy_at = -1
    return log_parsing_results

In [ ]:
# TODO add check if initial state has single negative frequency --> sanity check. 
# If so, it should not be very different eventual saddle-point. An IRC EQs should match. 

In [4]:
#pprint(log_parsing_results, sort_dicts=False)

In [5]:
def assemble_xyz(z: list, pos: np.array) -> str:
    """Assembling atomic numbers and positions into xyz format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    natoms =len(z)
    xyz = f"{natoms}\n\n"
    for _z, _pos in zip(z, pos): #.numpy()):
        xyz += f"{_z}\t" + "\t".join([str(x) for x in _pos]) + "\n"
    return xyz

In [6]:
def assemble_com(geom: list) -> str:
    """Assembling atomic numbers and positions into .com format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    coords_block = "\n".join(["\t".join([str(y) for y in x]) for x in geom])+"\n"
    return f'''# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
{coords_block.rstrip()}
Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200
'''

In [7]:
log_parsing_results = parse_grrm_saddlepoint_irc_log("scratch/C6H6_saddle_validation/C6H6-val_new_model_example.log")

com_block = assemble_com(log_parsing_results["initial_TS_geometry"])
print(com_block)

Found optimized structure and energy.
Saddle point was found after 37 iterations.
# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.138552417236	-1.289524234236	-0.760765290175
C	-0.574618715979	1.148561080488	0.848903484648
C	-0.43944556591	-1.352736986731	0.505187085052
C	0.234280555557	-0.052968814402	-1.404325848362
C	0.479719601914	0.892268841321	-0.05462727736
C	0.131728475445	-0.05383297278	1.171503824644
H	-1.643814414939	1.340358162814	0.700959548486
H	1.158765535257	-0.050997884502	-2.030023589872
H	-0.949802199629	-2.112967528744	1.108241036124
H	0.881048691403	-0.246864701457	1.942075259587
H	1.441659302798	1.397399397068	-0.0545689843
H	-0.580968848681	0.38130564116	-1.972559248471
Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200



In [16]:
pprint(log_parsing_results, sort_dicts=False)

{'initial_TS_energy': -231.852862902136,
 'initial_TS_geometry': [['C',
                          -0.138552417236,
                          -1.289524234236,
                          -0.760765290175],
                         ['C', -0.574618715979, 1.148561080488, 0.848903484648],
                         ['C', -0.43944556591, -1.352736986731, 0.505187085052],
                         ['C',
                          0.234280555557,
                          -0.052968814402,
                          -1.404325848362],
                         ['C', 0.479719601914, 0.892268841321, -0.05462727736],
                         ['C', 0.131728475445, -0.05383297278, 1.171503824644],
                         ['H', -1.643814414939, 1.340358162814, 0.700959548486],
                         ['H',
                          1.158765535257,
                          -0.050997884502,
                          -2.030023589872],
                         ['H',
                          -0.949802199629,
 

In [8]:
#with open("scratch/C6H6_saddle_validation/recreation-test-C6H6_irc-val_new_model_example.com", "w") as f:
#    f.write(com_block)

In [9]:
!cat scratch/C6H6_saddle_validation/C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C       -0.1385524172364524     -1.2895242342355837     -0.7607652901753402
C       -0.5746187159788264     1.1485610804884538      0.848903484647897
C       -0.43944556590990613    -1.3527369867314836     0.5051870850523872
C       0.23428055555705848     -0.05296881440221325    -1.4043258483621284
C       0.47971960191411184     0.8922688413213409      -0.054627277359575295
C       0.1317284754451082      -0.05383297277965539    1.1715038246435192
H       -1.6438144149385339     1.3403581628136956      0.7009595484863388
H       1.1587655352566293      -0.050997884501638606   -2.0300235898723886
H       -0.9498021996291618     -2.1129675287441425     1.1082410361243584
H       0.8810486914032343      -0.2468647014568118     1.942075259586514
H       1.441659302798145       1.397399397067585       -0.05456898430019256
H       -0.5809688486814069     0.3813056411604537      -1.9725592484713896
Options
Saddle+IRC
Do

In [10]:
!cat scratch/C6H6_saddle_validation/recreation-test-C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.138552417236	-1.289524234236	-0.760765290175
C	-0.574618715979	1.148561080488	0.848903484648
C	-0.43944556591	-1.352736986731	0.505187085052
C	0.234280555557	-0.052968814402	-1.404325848362
C	0.479719601914	0.892268841321	-0.05462727736
C	0.131728475445	-0.05383297278	1.171503824644
H	-1.643814414939	1.340358162814	0.700959548486
H	1.158765535257	-0.050997884502	-2.030023589872
H	-0.949802199629	-2.112967528744	1.108241036124
H	0.881048691403	-0.246864701457	1.942075259587
H	1.441659302798	1.397399397068	-0.0545689843
H	-0.580968848681	0.38130564116	-1.972559248471

Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200


In [11]:
import pymatgen

In [12]:
import os

from ase.io import read, write

generated_val_instances_dir = os.path.abspath("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/")

In [13]:
from ase.io import read
import io

def read_xyz_with_raw_comments(filename):
    """
    Reads an XYZ trajectory and treats the second line of each frame 
    strictly as a single string comment.
    """
    with open(filename, 'r') as f:
        while True:
            # 1. Read the number of atoms
            line = f.readline()
            if not line: break
            natoms = int(line.strip())
            
            # 2. Grab the entire comment line as a single string
            raw_comment = f.readline().strip()
            
            # 3. Read the coordinate block
            coords_lines = [f.readline() for _ in range(natoms)]
            
            # 4. Use io.StringIO to let ASE parse only the coordinates
            xyz_data = f"{natoms}\n{raw_comment}\n" + "".join(coords_lines)
            atoms = read(io.StringIO(xyz_data), format='xyz')
            
            # 5. Force the comment into the info dict as a single value
            atoms.info["comment"] = raw_comment
            
            yield atoms

In [14]:
files = [f for f in os.listdir(generated_val_instances_dir) if os.path.isfile(os.path.join(generated_val_instances_dir, f))]

for file in files:
    file_path = os.path.join(generated_val_instances_dir, file)
    temp_atoms = [atoms for atoms in read_xyz_with_raw_comments(file_path)]
    ts_name = file.replace(" ", "_").replace("(","_").replace(")","_").replace(",","-")[:-4]
    print(ts_name)
    instance_to_convert = temp_atoms[-1]
    #print(instance_to_convert.info["comment"])
    comment = instance_to_convert.info["comment"]
    model_name = "_".join(comment.split("/")[-2:])[:-6].replace(".", "_")
    new_dir = os.path.join(generated_val_instances_dir, "com-"+model_name)
    os.makedirs(new_dir, exist_ok=True)
    symbols = list(instance_to_convert.symbols)
    positions = instance_to_convert.positions
    geom_block = [[sym]+[str(x) for x in row] for sym, row in zip(symbols, positions)]
    com_block = assemble_com(geom_block)
    with open(os.path.join(new_dir, ts_name+".com"), "w") as f_out:
        f_out.write(com_block)
    print(os.path.join(new_dir, ts_name+".com"))

C6H6_5710-TS165_CON_24-23_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS165_CON_24-23_.com
C6H6_5710-TS282_CON_29-37_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS282_CON_29-37_.com
C6H6_5710-TS287_CON_86-123_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/C6H6_5710-TS287_CON_86-123_.com
C6H6_5710-TS1558_CON_464-100_
/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/com-C6H6

## 01/07/2026

Cleaned up a bit, made the log parsing loop into a per-log-file function. 

In [15]:
log_files_dir = Path("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/log-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/") # UPDATE

log_files = [f for f in os.listdir(log_files_dir) if os.path.isfile(os.path.join(log_files_dir, f))]
len(log_files)

FileNotFoundError: [Errno 2] No such file or directory: '/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/log-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68'

### 1. Compare one log file's results to its respective reference data. 



In [ ]:
log_file = log_files[6]
log_parsing_results = parse_grrm_saddlepoint_irc_log(log_files_dir / log_file)

parsed = parse("{network_name}-TS{ts_id_num}_CON_{eq_id_1}-{eq_id_2}_.log", log_file)
network_name = parsed['network_name'] 
ts_id_num  = parsed['ts_id_num']
eq_id_1 = parsed['eq_id_1']
eq_id_2 = parsed['eq_id_2']
ts_xyz_file = f"{network_name}-TS{ts_id_num} CON({eq_id_1},{eq_id_2}).xyz"
ref_atoms = read(Path(generated_val_instances_dir) / ts_xyz_file, index=":")[0:3]
len(ref_atoms)



In [ ]:
pprint(log_parsing_results, sort_dicts=False)

In [ ]:
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

if log_parsing_results['saddle_point_found']:
    reopt_ts_geometry = log_parsing_results['reopt_TS_geometry']
    reopt_ts_species = [x[0] for x in reopt_ts_geometry]
    reopt_ts_coords = [x[1:] for x in reopt_ts_geometry]
    
    mol1 = Molecule(reopt_ts_species, reopt_ts_coords)
    
    ref_ts_atom = ref_atoms[1]
    mol2 = Molecule(list(ref_ts_atom.symbols), ref_ts_atom.positions)

# 1. Initialize with the target (mol2)
# The matcher now knows the reference structure it needs to map to
matcher = GeneticOrderMatcher(mol2)

# 2. Fit the input molecule (mol1) to the target (mol2)
try:
    # fit returns (aligned_mol, rmsd)
    _, rmsd = matcher.fit(mol1)
    print(f"Coordinate-based RMSD: {rmsd} Å")
except Exception as e:
    print(f"Could not calculate RMSD: {e}")

In [17]:
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

# Checking for EQ matches:
if log_parsing_results['1st_irc_EQ_geometry'] is not None and log_parsing_results['2nd_irc_EQ_geometry'] is not None:
    irc_eq1_species = [x[0] for x in log_parsing_results['1st_irc_EQ_geometry']]
    irc_eq1_coords = [x[1:] for x in log_parsing_results['1st_irc_EQ_geometry']]
    irc_eq1_mol = Molecule(irc_eq1_species, irc_eq1_coords)
    irc_eq2_species = [x[0] for x in log_parsing_results['2nd_irc_EQ_geometry']]
    irc_eq2_coords = [x[1:] for x in log_parsing_results['2nd_irc_EQ_geometry']]
    irc_eq2_mol = Molecule(irc_eq2_species, irc_eq2_coords)
    
    ref_eq1_mol = Molecule(list(ref_atoms[0].symbols), ref_atoms[0].positions)
    ref_eq2_mol = Molecule(list(ref_atoms[2].symbols), ref_atoms[2].positions)
    
    # 1. Initialize with the target (mol2)
    # The matcher now knows the reference structure it needs to map to
    eq1_matcher = GeneticOrderMatcher(ref_eq1_mol)
    eq2_matcher = GeneticOrderMatcher(ref_eq2_mol)
    
    # 2. Fit the input molecule (mol1) to the target (mol2)
    for i, irc_eq in enumerate([irc_eq1_mol, irc_eq2_mol]):
        print(f"Checking IRC EQ number {i}:")
        try:
            # fit returns (aligned_mol, rmsd)
            _, rmsd = eq1_matcher.fit(irc_eq)
            print(f"Coordinate-based RMSD on first ref EQ: {rmsd} Å")
        except Exception as e:
            print(f"Could not calculate RMSD on first ref EQ: {e}")
        try:
            # fit returns (aligned_mol, rmsd)
            _, rmsd = eq2_matcher.fit(irc_eq)
            print(f"Coordinate-based RMSD on second ref EQ: {rmsd} Å")
        except Exception as e:
            print(f"Could not calculate RMSD on second ref EQ: {e}")
        print("")
else:
    print("Missing EQs after IRC calculation.")

Missing EQs after IRC calculation.


### 2. Loop over all the log files in directory, compile stats on the results

In [ ]:
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

for i, log_file in enumerate(log_files):
    parsed = parse("{network_name}-TS{ts_id_num}_CON_{eq_id_1}-{eq_id_2}_.log", log_file)
    network_name = parsed['network_name'] 
    ts_id_num  = parsed['ts_id_num']
    eq_id_1 = parsed['eq_id_1']
    eq_id_2 = parsed['eq_id_2']
    ts_xyz_file = f"{network_name}-TS{ts_id_num} CON({eq_id_1},{eq_id_2}).xyz"
    print(f"{i}: {ts_xyz_file[:-4]}")
    log_parsing_results = parse_grrm_saddlepoint_irc_log(log_files_dir / log_file)
    ref_atoms = read(Path(generated_val_instances_dir) / ts_xyz_file, index=":")[0:3]

    # Checking for EQ matches:
    if log_parsing_results['1st_irc_EQ_geometry'] is not None and log_parsing_results['2nd_irc_EQ_geometry'] is not None:
        irc_eq1_species = [x[0] for x in log_parsing_results['1st_irc_EQ_geometry']]
        irc_eq1_coords = [x[1:] for x in log_parsing_results['1st_irc_EQ_geometry']]
        irc_eq1_mol = Molecule(irc_eq1_species, irc_eq1_coords)
        irc_eq2_species = [x[0] for x in log_parsing_results['2nd_irc_EQ_geometry']]
        irc_eq2_coords = [x[1:] for x in log_parsing_results['2nd_irc_EQ_geometry']]
        irc_eq2_mol = Molecule(irc_eq2_species, irc_eq2_coords)
        
        ref_eq1_mol = Molecule(list(ref_atoms[0].symbols), ref_atoms[0].positions)
        ref_eq2_mol = Molecule(list(ref_atoms[2].symbols), ref_atoms[2].positions)
        
        # 1. Initialize with the target (mol2)
        # The matcher now knows the reference structure it needs to map to
        eq1_matcher = GeneticOrderMatcher(ref_eq1_mol)
        eq2_matcher = GeneticOrderMatcher(ref_eq2_mol)
        
        # 2. Fit the input molecule (mol1) to the target (mol2)
        for j, irc_eq in enumerate([irc_eq1_mol, irc_eq2_mol]):
            print(f"Checking IRC EQ number {i}:")
            try:
                # fit returns (aligned_mol, rmsd)
                _, rmsd = eq1_matcher.fit(irc_eq)
                print(f"Coordinate-based RMSD on first ref EQ: {rmsd} Å")
            except Exception as e:
                print(f"Could not calculate RMSD on first ref EQ: {e}")
            try:
                # fit returns (aligned_mol, rmsd)
                _, rmsd = eq2_matcher.fit(irc_eq)
                print(f"Coordinate-based RMSD on second ref EQ: {rmsd} Å")
            except Exception as e:
                print(f"Could not calculate RMSD on second ref EQ: {e}")
    else:
        print("Missing EQs after IRC calculation.\n")

In [ ]:
# TODO also compare with -1* EQ state, take the min RMSD one.